In [1]:
import pandas as pd
import glob
import os
import re

In [2]:
# Load all CSV files starting with "500" from the cdc_data_raw subfolder
file_pattern = os.path.join("cdc_data_raw", "500*.csv")
files = glob.glob(file_pattern)

print(f"Found {len(files)} files:")
for f in files:
    print(f"  {f}")

# Load and combine all files into one dataframe
cdc_df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print(f"\nCombined dataframe shape: {cdc_df.shape}")
cdc_df.head()

Found 4 files:
  cdc_data_raw/500_Cities__Local_Data_for_Better_Health,_2019_release_20260406.csv
  cdc_data_raw/500_Cities__Local_Data_for_Better_Health,_2016_release_20260406.csv
  cdc_data_raw/500_Cities__Local_Data_for_Better_Health,_2017_release_20260406.csv
  cdc_data_raw/500_Cities__Local_Data_for_Better_Health,_2018_release_20260406.csv

Combined dataframe shape: (3240412, 26)


,Year,StateAbbr,StateDesc,CityName,GeographicLevel,DataSource,Category,UniqueID,Measure,Data_Value_Unit,...,Data_Value_Footnote,PopulationCount,GeoLocation,CategoryID,MeasureId,CityFIPS,TractFIPS,Short_Question_Text,Population2010,Geolocation
0,2017,CA,California,Hawthorne,Census Tract,BRFSS,Health Outcomes,0632548-06037602504,Arthritis among adults aged >=18 Years,%,...,NaN,"4,407","(33.905547923, -118.337332298)",HLTHOUT,ARTHRITIS,632548.0,6.037603e+09,Arthritis,NaN,NaN
1,2017,CA,California,Hawthorne,City,BRFSS,Unhealthy Behaviors,0632548,Current smoking among adults aged >=18 Years,%,...,NaN,"84,293","(33.914667701, -118.347667728)",UNHBEH,CSMOKING,632548.0,NaN,Current Smoking,NaN,NaN
2,2017,CA,California,Hayward,City,BRFSS,Health Outcomes,0633000,Coronary heart disease among adults aged >=18 ...,%,...,NaN,"144,186","(37.6329591551, -122.077051051)",HLTHOUT,CHD,633000.0,NaN,Coronary Heart Disease,NaN,NaN
3,2017,CA,California,Hayward,City,BRFSS,Unhealthy Behaviors,0633000,Obesity among adults aged >=18 Years,%,...,NaN,"144,186","(37.6329591551, -122.077051051)",UNHBEH,OBESITY,633000.0,NaN,Obesity,NaN,NaN
4,2017,CA,California,Hemet,City,BRFSS,Prevention,0633182,Cholesterol screening among adults aged >=18 Y...,%,...,NaN,"78,657","(33.7352277311, -116.994605005)",PREVENT,CHOLSCREEN,633182.0,NaN,Cholesterol Screening,NaN,NaN


In [3]:
cdc_df = cdc_df[cdc_df["GeographicLevel"] == "Census Tract"]

# Filter MeasureId to only include relevant health measures
measures_to_keep = ["BPHIGH", "CASTHMA", "COPD", "MHLTH", "PHLTH", "SLEEP", "STROKE"]
cdc_df = cdc_df[cdc_df["MeasureId"].isin(measures_to_keep)]

print(f"Filtered dataframe shape: {cdc_df.shape}")
cdc_df.head()

Filtered dataframe shape: (784112, 26)


,Year,StateAbbr,StateDesc,CityName,GeographicLevel,DataSource,Category,UniqueID,Measure,Data_Value_Unit,...,Data_Value_Footnote,PopulationCount,GeoLocation,CategoryID,MeasureId,CityFIPS,TractFIPS,Short_Question_Text,Population2010,Geolocation
35,2017,AZ,Arizona,Phoenix,Census Tract,BRFSS,Health Outcomes,0455000-04013116800,Stroke among adults aged >=18 Years,%,...,NaN,"2,655","(33.465854512, -112.108603053)",HLTHOUT,STROKE,455000.0,4.013117e+09,Stroke,NaN,NaN
39,2017,AZ,Arizona,Tucson,Census Tract,BRFSS,Health Outcomes,0477000-04019004710,High blood pressure among adults aged >=18 Years,%,...,NaN,"2,138","(32.2851840496, -110.941404638)",HLTHOUT,BPHIGH,477000.0,4.019005e+09,High Blood Pressure,NaN,NaN
91,2017,CA,California,Menifee,Census Tract,BRFSS,Health Outcomes,0646842-06065042740,High blood pressure among adults aged >=18 Years,%,...,NaN,"2,017","(33.7251014333, -117.197756929)",HLTHOUT,BPHIGH,646842.0,6.065043e+09,High Blood Pressure,NaN,NaN
126,2017,CA,California,Redlands,Census Tract,BRFSS,Health Outcomes,0659962-06071008601,Stroke among adults aged >=18 Years,%,...,NaN,56,"(34.051688893, -117.137702007)",HLTHOUT,STROKE,659962.0,6.071009e+09,Stroke,NaN,NaN
140,2017,CA,California,Sacramento,Census Tract,BRFSS,Health Outcomes,0664000-06067009634,Stroke among adults aged >=18 Years,%,...,NaN,"4,913","(38.4697876124, -121.44091893)",HLTHOUT,STROKE,664000.0,6.067010e+09,Stroke,NaN,NaN


In [4]:
census_tracts = pd.read_csv(os.path.join("cdc_data_raw", "CensusTractList2022.csv"))

census_tracts["CountyFIPS"] = (
    census_tracts["State code"].astype(str).str.zfill(2) +
    census_tracts["County code"].astype(str).str.zfill(3)
)

print(f"Census tract list shape: {census_tracts.shape}")
census_tracts.head()

Census tract list shape: (85528, 15)


,Year,MSA/MD code type,MSA/MD code,State code,County code,Tract,MSA/MD name,State,County name,FIPS code,MSA/MD MFI,Tract MFI,Tract income percentage,Tract income level,CountyFIPS
0,2022,MSA,10180,48,59,30101,"ABILENE, TX ...",TX,CALLAHAN COUNTY ...,48059030101,68388,61923.0,90.54,Middle,48059
1,2022,MSA,10180,48,59,30102,"ABILENE, TX ...",TX,CALLAHAN COUNTY ...,48059030102,68388,66132.0,96.70,Middle,48059
2,2022,MSA,10180,48,59,30200,"ABILENE, TX ...",TX,CALLAHAN COUNTY ...,48059030200,68388,59531.0,87.04,Middle,48059
3,2022,MSA,10180,48,253,20101,"ABILENE, TX ...",TX,JONES COUNTY ...,48253020101,68388,55179.0,80.68,Middle,48253
4,2022,MSA,10180,48,253,20102,"ABILENE, TX ...",TX,JONES COUNTY ...,48253020102,68388,0.0,0.00,Unknown,48253


In [5]:
# Merge cdc_df with census_tracts to add county name
cdc_df = cdc_df.merge(
    census_tracts[["FIPS code", "County name", "CountyFIPS"]],
    left_on="TractFIPS",
    right_on="FIPS code",
    how="left"
).drop(columns="FIPS code")

print(f"Merged dataframe shape: {cdc_df.shape}")
print(f"Rows with missing county: {cdc_df["County name"].isna().sum()}")
cdc_df.head()

Merged dataframe shape: (784112, 28)
Rows with missing county: 118412


,Year,StateAbbr,StateDesc,CityName,GeographicLevel,DataSource,Category,UniqueID,Measure,Data_Value_Unit,...,GeoLocation,CategoryID,MeasureId,CityFIPS,TractFIPS,Short_Question_Text,Population2010,Geolocation,County name,CountyFIPS
0,2017,AZ,Arizona,Phoenix,Census Tract,BRFSS,Health Outcomes,0455000-04013116800,Stroke among adults aged >=18 Years,%,...,"(33.465854512, -112.108603053)",HLTHOUT,STROKE,455000.0,4.013117e+09,Stroke,NaN,NaN,MARICOPA COUNTY ...,04013
1,2017,AZ,Arizona,Tucson,Census Tract,BRFSS,Health Outcomes,0477000-04019004710,High blood pressure among adults aged >=18 Years,%,...,"(32.2851840496, -110.941404638)",HLTHOUT,BPHIGH,477000.0,4.019005e+09,High Blood Pressure,NaN,NaN,PIMA COUNTY ...,04019
2,2017,CA,California,Menifee,Census Tract,BRFSS,Health Outcomes,0646842-06065042740,High blood pressure among adults aged >=18 Years,%,...,"(33.7251014333, -117.197756929)",HLTHOUT,BPHIGH,646842.0,6.065043e+09,High Blood Pressure,NaN,NaN,RIVERSIDE COUNTY ...,06065
3,2017,CA,California,Redlands,Census Tract,BRFSS,Health Outcomes,0659962-06071008601,Stroke among adults aged >=18 Years,%,...,"(34.051688893, -117.137702007)",HLTHOUT,STROKE,659962.0,6.071009e+09,Stroke,NaN,NaN,SAN BERNARDINO COUNTY ...,06071
4,2017,CA,California,Sacramento,Census Tract,BRFSS,Health Outcomes,0664000-06067009634,Stroke among adults aged >=18 Years,%,...,"(38.4697876124, -121.44091893)",HLTHOUT,STROKE,664000.0,6.067010e+09,Stroke,NaN,NaN,SACRAMENTO COUNTY ...,06067


In [6]:
# Convert to numeric, coercing errors to NaN
cdc_df["Data_Value"] = pd.to_numeric(cdc_df["Data_Value"], errors="coerce")
cdc_df["PopulationCount"] = pd.to_numeric(cdc_df["PopulationCount"], errors="coerce")

# Fill missing PopulationCount with 1 so they still contribute to the weighted average
cdc_df["PopulationCount"] = cdc_df["PopulationCount"].fillna(1)

# Calculate weighted average of Data_Value by County name, Year, StateAbbr, MeasureId
cdc_county = cdc_df.groupby(["Year", "StateAbbr", "MeasureId", "County name", "CountyFIPS"]).apply(
    lambda x: pd.Series({
        "Data_Value": (
            (x["Data_Value"] * x["PopulationCount"]).sum() / x["PopulationCount"].sum()
            if x["PopulationCount"].sum() > 0
            else None
        )
    })
).reset_index()

print(f"Grouped dataframe shape: {cdc_county.shape}")
print(f"Unique counties: {cdc_county['County name'].nunique()}")
print(f"Rows with missing Data_Value: {cdc_county['Data_Value'].isna().sum()}")
cdc_county.head()

Grouped dataframe shape: (8125, 6)
Unique counties: 288
Rows with missing Data_Value: 0


,Year,StateAbbr,MeasureId,County name,CountyFIPS,Data_Value
0,2013,AK,BPHIGH,ANCHORAGE MUNICIPALITY ...,02020,28.691667
1,2013,AL,BPHIGH,JEFFERSON COUNTY ...,01073,41.893750
2,2013,AL,BPHIGH,MADISON COUNTY ...,01089,37.654348
3,2013,AL,BPHIGH,MOBILE COUNTY ...,01097,44.833333
4,2013,AL,BPHIGH,MONTGOMERY COUNTY ...,01101,39.792727


In [7]:
# Load all CSV files starting with "PLACES" from the cdc_data_raw subfolder
file_pattern = os.path.join("cdc_data_raw", "PLACES*.csv")
files = glob.glob(file_pattern)

print(f"Found {len(files)} files:")
for f in files:
    print(f"  {f}")

# Load and combine all files, extracting year from filename
dfs = []
for f in files:
    df = pd.read_csv(f)
    # Extract 4-digit year before "_release" in filename and subtract 2
    year = int(re.search(r'(\d{4})_release', os.path.basename(f)).group(1)) - 2
    df["Year"] = year
    dfs.append(df)

places_df = pd.concat(dfs, ignore_index=True)

print(f"\nCombined dataframe shape: {places_df.shape}")
print(f"Unique years: {sorted(places_df['Year'].unique())}")
places_df.head()

Found 6 files:
  cdc_data_raw/PLACES__County_Data_(GIS_Friendly_Format),_2025_release_20260406.csv
  cdc_data_raw/PLACES__County_Data_(GIS_Friendly_Format),_2020_release_20260406.csv
  cdc_data_raw/PLACES__County_Data_(GIS_Friendly_Format),_2024_release_20260406.csv
  cdc_data_raw/PLACES__County_Data_(GIS_Friendly_Format),_2021_release_20260406.csv
  cdc_data_raw/PLACES__County_Data_(GIS_Friendly_Format),_2022_release_20260406.csv
  cdc_data_raw/PLACES__County_Data_(GIS_Friendly_Format),_2023_release_20260406.csv

Combined dataframe shape: (18857, 188)
Unique years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


,StateAbbr,StateDesc,CountyName,CountyFIPS,TotalPopulation,TotalPop18plus,ACCESS2_CrudePrev,ACCESS2_Crude95CI,ACCESS2_AdjPrev,ACCESS2_Adj95CI,...,COREW_AdjPrev,COREW_Adj95CI,KIDNEY_CrudePrev,KIDNEY_Crude95CI,KIDNEY_AdjPrev,KIDNEY_Adj95CI,ISOLATION_CrudePrev,ISOLATION_Crude95CI,ISOLATION_AdjPrev,ISOLATION_Adj95CI
0,KY,Kentucky,Trigg,21221,"14,369","11,280",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,KY,Kentucky,Wayne,21231,"19,580","15,608",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,KY,Kentucky,Washington,21229,"12,267","9,420",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PA,Pennsylvania,Perry,42099,"46,083","36,495",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,PA,Pennsylvania,Venango,42121,"49,431","39,983",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
print(places_df.columns.tolist())
print(f"Year column position: {places_df.columns.tolist().index('Year')}")

['StateAbbr', 'StateDesc', 'CountyName', 'CountyFIPS', 'TotalPopulation', 'TotalPop18plus', 'ACCESS2_CrudePrev', 'ACCESS2_Crude95CI', 'ACCESS2_AdjPrev', 'ACCESS2_Adj95CI', 'ARTHRITIS_CrudePrev', 'ARTHRITIS_Crude95CI', 'ARTHRITIS_AdjPrev', 'ARTHRITIS_Adj95CI', 'BINGE_CrudePrev', 'BINGE_Crude95CI', 'BINGE_AdjPrev', 'BINGE_Adj95CI', 'BPHIGH_CrudePrev', 'BPHIGH_Crude95CI', 'BPHIGH_AdjPrev', 'BPHIGH_Adj95CI', 'BPMED_CrudePrev', 'BPMED_Crude95CI', 'BPMED_AdjPrev', 'BPMED_Adj95CI', 'CANCER_CrudePrev', 'CANCER_Crude95CI', 'CANCER_AdjPrev', 'CANCER_Adj95CI', 'CASTHMA_CrudePrev', 'CASTHMA_Crude95CI', 'CASTHMA_AdjPrev', 'CASTHMA_Adj95CI', 'CHD_CrudePrev', 'CHD_Crude95CI', 'CHD_AdjPrev', 'CHD_Adj95CI', 'CHECKUP_CrudePrev', 'CHECKUP_Crude95CI', 'CHECKUP_AdjPrev', 'CHECKUP_Adj95CI', 'CHOLSCREEN_CrudePrev', 'CHOLSCREEN_Crude95CI', 'CHOLSCREEN_AdjPrev', 'CHOLSCREEN_Adj95CI', 'COLON_SCREEN_CrudePrev', 'COLON_SCREEN_Crude95CI', 'COLON_SCREEN_AdjPrev', 'COLON_SCREEN_Adj95CI', 'COPD_CrudePrev', 'COPD_Crud

In [9]:
# Add leading 0 to CountyFIPS in places_df if length is 4
places_df["CountyFIPS"] = places_df["CountyFIPS"].astype(str).str.zfill(5)

# Merge places_df with census_tracts on CountyFIPS to add County name
places_df = places_df.merge(
    census_tracts[["CountyFIPS", "County name"]].drop_duplicates(),
    on="CountyFIPS",
    how="left"
)

print(f"Merged dataframe shape: {places_df.shape}")
print(f"Rows with missing county name: {places_df['County name'].isna().sum()}")
places_df[["CountyFIPS", "County name"]].head()

Merged dataframe shape: (18857, 189)
Rows with missing county name: 20


,CountyFIPS,County name
0,21221,TRIGG COUNTY ...
1,21231,WAYNE COUNTY ...
2,21229,WASHINGTON COUNTY ...
3,42099,PERRY COUNTY ...
4,42121,VENANGO COUNTY ...


In [10]:
# Define id columns explicitly
id_cols = ["StateAbbr", "StateDesc", "CountyName", "CountyFIPS", "TotalPopulation", "TotalPop18plus", "Geolocation", "Year", "County name"]

# Only keep id_cols that actually exist in the dataframe
id_cols = [col for col in id_cols if col in places_df.columns]

# All other columns become measure columns
measure_cols = [col for col in places_df.columns if col not in id_cols]

# Pivot measure columns from wide to long format
places_df = places_df.melt(
    id_vars=id_cols,
    value_vars=measure_cols,
    var_name="MeasureId",
    value_name="Data_Value"
)

print(f"Pivoted dataframe shape: {places_df.shape}")
print(f"Columns: {places_df.columns.tolist()}")
print(f"Unique years: {sorted(places_df['Year'].unique())}")
places_df.head()

Pivoted dataframe shape: (3394260, 11)
Columns: ['StateAbbr', 'StateDesc', 'CountyName', 'CountyFIPS', 'TotalPopulation', 'TotalPop18plus', 'Geolocation', 'Year', 'County name', 'MeasureId', 'Data_Value']
Unique years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


,StateAbbr,StateDesc,CountyName,CountyFIPS,TotalPopulation,TotalPop18plus,Geolocation,Year,County name,MeasureId,Data_Value
0,KY,Kentucky,Trigg,21221,"14,369","11,280",POINT (-87.8732998555619 36.8063370344073),2023,TRIGG COUNTY ...,ACCESS2_CrudePrev,NaN
1,KY,Kentucky,Wayne,21231,"19,580","15,608",POINT (-84.8283531236799 36.8008845409014),2023,WAYNE COUNTY ...,ACCESS2_CrudePrev,NaN
2,KY,Kentucky,Washington,21229,"12,267","9,420",POINT (-85.175051728938 37.7532982899311),2023,WASHINGTON COUNTY ...,ACCESS2_CrudePrev,NaN
3,PA,Pennsylvania,Perry,42099,"46,083","36,495",POINT (-77.2622712474235 40.3984083146619),2023,PERRY COUNTY ...,ACCESS2_CrudePrev,NaN
4,PA,Pennsylvania,Venango,42121,"49,431","39,983",POINT (-79.7581868804174 41.4006288227586),2023,VENANGO COUNTY ...,ACCESS2_CrudePrev,NaN


In [11]:
# Split MeasureId into MeasureId and Value_Type using the last underscore as delimiter
places_df[["MeasureId", "Value_Type"]] = places_df["MeasureId"].str.rsplit("_", n=1, expand=True)

print(f"Unique MeasureIds: {places_df['MeasureId'].nunique()}")
print(f"Unique Value_Types: {places_df['Value_Type'].unique()}")
places_df[["MeasureId", "Value_Type", "Data_Value"]].head()

Unique MeasureIds: 45
Unique Value_Types: <StringArray>
['CrudePrev', 'Crude95CI', 'AdjPrev', 'Adj95CI', 'Crude95ci']
Length: 5, dtype: str


,MeasureId,Value_Type,Data_Value
0,ACCESS2,CrudePrev,NaN
1,ACCESS2,CrudePrev,NaN
2,ACCESS2,CrudePrev,NaN
3,ACCESS2,CrudePrev,NaN
4,ACCESS2,CrudePrev,NaN


In [12]:
# Filter MeasureId to only include relevant health measures
places_df = places_df[places_df["MeasureId"].isin(measures_to_keep)]

print(f"Filtered dataframe shape: {places_df.shape}")
print(f"Unique MeasureIds: {places_df['MeasureId'].unique()}")
places_df.head()

Filtered dataframe shape: (527996, 12)
Unique MeasureIds: <StringArray>
['BPHIGH', 'CASTHMA', 'COPD', 'MHLTH', 'PHLTH', 'SLEEP', 'STROKE']
Length: 7, dtype: str


,StateAbbr,StateDesc,CountyName,CountyFIPS,TotalPopulation,TotalPop18plus,Geolocation,Year,County name,MeasureId,Data_Value,Value_Type
226284,KY,Kentucky,Trigg,21221,"14,369","11,280",POINT (-87.8732998555619 36.8063370344073),2023,TRIGG COUNTY ...,BPHIGH,NaN,CrudePrev
226285,KY,Kentucky,Wayne,21231,"19,580","15,608",POINT (-84.8283531236799 36.8008845409014),2023,WAYNE COUNTY ...,BPHIGH,NaN,CrudePrev
226286,KY,Kentucky,Washington,21229,"12,267","9,420",POINT (-85.175051728938 37.7532982899311),2023,WASHINGTON COUNTY ...,BPHIGH,NaN,CrudePrev
226287,PA,Pennsylvania,Perry,42099,"46,083","36,495",POINT (-77.2622712474235 40.3984083146619),2023,PERRY COUNTY ...,BPHIGH,NaN,CrudePrev
226288,PA,Pennsylvania,Venango,42121,"49,431","39,983",POINT (-79.7581868804174 41.4006288227586),2023,VENANGO COUNTY ...,BPHIGH,NaN,CrudePrev


In [13]:
# Check columns in cdc_county
print("cdc_county columns:", cdc_county.columns.tolist())
print("places_df columns:", places_df.columns.tolist())

cdc_county columns: ['Year', 'StateAbbr', 'MeasureId', 'County name', 'CountyFIPS', 'Data_Value']
places_df columns: ['StateAbbr', 'StateDesc', 'CountyName', 'CountyFIPS', 'TotalPopulation', 'TotalPop18plus', 'Geolocation', 'Year', 'County name', 'MeasureId', 'Data_Value', 'Value_Type']


In [14]:
# Keep only columns that exist in cdc_county
common_cols = [col for col in cdc_county.columns if col in places_df.columns]
places_df_aligned = places_df[common_cols]

# Stack cdc_county and places_df
combined_df = pd.concat([cdc_county, places_df_aligned], ignore_index=True)

print(f"cdc_county shape: {cdc_county.shape}")
print(f"places_df_aligned shape: {places_df_aligned.shape}")
print(f"Combined dataframe shape: {combined_df.shape}")
print(f"Unique years: {sorted(combined_df['Year'].unique())}")
combined_df.head()

cdc_county shape: (8125, 6)
places_df_aligned shape: (527996, 6)
Combined dataframe shape: (536121, 6)
Unique years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


,Year,StateAbbr,MeasureId,County name,CountyFIPS,Data_Value
0,2013,AK,BPHIGH,ANCHORAGE MUNICIPALITY ...,02020,28.691667
1,2013,AL,BPHIGH,JEFFERSON COUNTY ...,01073,41.89375
2,2013,AL,BPHIGH,MADISON COUNTY ...,01089,37.654348
3,2013,AL,BPHIGH,MOBILE COUNTY ...,01097,44.833333
4,2013,AL,BPHIGH,MONTGOMERY COUNTY ...,01101,39.792727


In [17]:
# Convert Data_Value to numeric before pivoting
combined_df["Data_Value"] = pd.to_numeric(combined_df["Data_Value"], errors="coerce")

# Pivot combined_df so each MeasureId becomes its own column
combined_pivot = combined_df.pivot_table(
    index=["Year", "StateAbbr", "County name", "CountyFIPS"],
    columns="MeasureId",
    values="Data_Value",
    aggfunc="mean"
).reset_index()

# Flatten column names
combined_pivot.columns.name = None

print(f"Pivoted dataframe shape: {combined_pivot.shape}")
print(f"Columns: {combined_pivot.columns.tolist()}")
print(f"Unique years: {sorted(combined_pivot['Year'].unique())}")
combined_pivot.head()

Pivoted dataframe shape: (20462, 11)
Columns: ['Year', 'StateAbbr', 'County name', 'CountyFIPS', 'BPHIGH', 'CASTHMA', 'COPD', 'MHLTH', 'PHLTH', 'SLEEP', 'STROKE']
Unique years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


,Year,StateAbbr,County name,CountyFIPS,BPHIGH,CASTHMA,COPD,MHLTH,PHLTH,SLEEP,STROKE
0,2013,AK,ANCHORAGE MUNICIPALITY ...,02020,28.691667,NaN,NaN,NaN,NaN,NaN,NaN
1,2013,AL,JEFFERSON COUNTY ...,01073,41.893750,NaN,NaN,NaN,NaN,NaN,NaN
2,2013,AL,MADISON COUNTY ...,01089,37.654348,NaN,NaN,NaN,NaN,NaN,NaN
3,2013,AL,MOBILE COUNTY ...,01097,44.833333,NaN,NaN,NaN,NaN,NaN,NaN
4,2013,AL,MONTGOMERY COUNTY ...,01101,39.792727,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
missing = combined_pivot.isnull().mean().mul(100).round(2).sort_values(ascending=False)
print(missing[missing > 0]) 

BPHIGH     4.96
SLEEP      4.76
CASTHMA    2.93
COPD       2.93
MHLTH      2.93
PHLTH      2.93
STROKE     2.93
dtype: float64


In [19]:
combined_pivot.to_csv("combined_cdc_final.csv", index=False)